# Демо 1. Класичне донавчання: зміна голови в моделі BERT

У цьому ноутбуці донавчаємо модель **BERT** для задачі **визначення парафраз**: модель отримує два речення й визначає, чи передають вони той самий зміст.

План:
1. Завантажити датасет MRPC і токенізувати пари речень.
2. Завантажити BERT із **новою головою**-класифікатором.
3. **Повне донавчання** з `Trainer`: тренуються всі ваги (кістяк і голова).
4. Перевірити модель на власних прикладах.

Приклад за мотивами розділу 3 курсу [Hugging Face LLM Course](https://huggingface.co/learn/llm-course/chapter3/1).
Для запуску в Colab оберіть середовище з GPU: *Runtime → Change runtime type → T4 GPU*.

### Підготовка середовища
Встановлення залежностей та вимірювання часу виконання. Оригінальні комірки демо залишено без змін.

In [1]:
import time
from IPython import get_ipython
from IPython.display import display

notebook_start = time.perf_counter()
cell_timings = []
ip = get_ipython()

def _lab5_pre(info):
    global _lab5_cell_start
    _lab5_cell_start = time.perf_counter()

def _lab5_post(result):
    if '_lab5_cell_start' in globals():
        err = result.error_before_exec or result.error_in_exec
        cell_timings.append({
            'cell': (result.info.raw_cell.strip().splitlines() or [''])[0][:75],
            'seconds': time.perf_counter() - _lab5_cell_start,
            'error': str(err) if err else ''
        })

if ip is not None:
    for event, callback in globals().get('_lab5_hooks', []):
        try:
            ip.events.unregister(event, callback)
        except ValueError:
            pass
    _lab5_hooks = [('pre_run_cell', _lab5_pre), ('post_run_cell', _lab5_post)]
    for event, callback in _lab5_hooks:
        ip.events.register(event, callback)

In [2]:
%pip install -q "transformers>=4.55,<5" "datasets>=3,<5" "accelerate>=1,<2"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [3]:
import numpy as np
import torch
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

In [4]:
# Фіксуємо seed, щоб результати повторювалися між запусками
set_seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Пристрій: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name()}")
else:
    print("Включіть GPU, бо набридне чекати ;)")

Пристрій: cuda
GPU: Tesla T4


## 1. Дані: MRPC

**MRPC** (Microsoft Research Paraphrase Corpus) — пари речень з новинних статей.
Мітка `1` означає, що речення є парафразами, мітка `0` — що ні.

In [5]:
raw_datasets = load_dataset("nyu-mll/glue", "mrpc")
raw_datasets

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

mrpc/train-00000-of-00001.parquet:   0%|          | 0.00/649k [00:00<?, ?B/s]

mrpc/validation-00000-of-00001.parquet:   0%|          | 0.00/75.7k [00:00<?, ?B/s]

mrpc/test-00000-of-00001.parquet:   0%|          | 0.00/308k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3668 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/408 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1725 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})

In [6]:
# Подивимося на один приклад
print("Приклад нееквівалентних речень:")
example = raw_datasets["train"][3]
print("Речення 1:", example["sentence1"])
print("Речення 2:", example["sentence2"])
print("Мітка:   ", example["label"], "→", raw_datasets["train"].features["label"].names[example["label"]])

print("")
print("Приклад еквівалентних речень:")
example = raw_datasets["train"][5]
print("Речення 1:", example["sentence1"])
print("Речення 2:", example["sentence2"])
print("Мітка:   ", example["label"], "→", raw_datasets["train"].features["label"].names[example["label"]])

Приклад нееквівалентних речень:
Речення 1: Around 0335 GMT , Tab shares were up 19 cents , or 4.4 % , at A $ 4.56 , having earlier set a record high of A $ 4.57 .
Речення 2: Tab shares jumped 20 cents , or 4.6 % , to set a record closing high at A $ 4.57 .
Мітка:    0 → not_equivalent

Приклад еквівалентних речень:
Речення 1: Revenue in the first quarter of the year dropped 15 percent from the same period a year earlier .
Речення 2: With the scandal hanging over Stewart 's company , revenue the first quarter of the year dropped 15 percent from the same period a year earlier .
Мітка:    1 → equivalent


### Знайомимось з BERT BASE UNCASED

Для цього прочитайте інформацію на [картці моделі](https://huggingface.co/google-bert/bert-base-uncased)

In [7]:
from transformers import pipeline

unmasker = pipeline('fill-mask', model='bert-base-uncased')
unmasker("The man worked as a [MASK].")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


[{'score': 0.09747552871704102,
  'token': 10533,
  'token_str': 'carpenter',
  'sequence': 'the man worked as a carpenter.'},
 {'score': 0.052383244037628174,
  'token': 15610,
  'token_str': 'waiter',
  'sequence': 'the man worked as a waiter.'},
 {'score': 0.049627017229795456,
  'token': 13362,
  'token_str': 'barber',
  'sequence': 'the man worked as a barber.'},
 {'score': 0.037886038422584534,
  'token': 15893,
  'token_str': 'mechanic',
  'sequence': 'the man worked as a mechanic.'},
 {'score': 0.037680864334106445,
  'token': 18968,
  'token_str': 'salesman',
  'sequence': 'the man worked as a salesman.'}]

In [8]:
print("Не по темі, але погляньте, яка \"несправедливість\"...")
unmasker("The woman worked as a [MASK].")

Не по темі, але погляньте, яка "несправедливість"...


[{'score': 0.21981368958950043,
  'token': 6821,
  'token_str': 'nurse',
  'sequence': 'the woman worked as a nurse.'},
 {'score': 0.15973998606204987,
  'token': 13877,
  'token_str': 'waitress',
  'sequence': 'the woman worked as a waitress.'},
 {'score': 0.11547233909368515,
  'token': 10850,
  'token_str': 'maid',
  'sequence': 'the woman worked as a maid.'},
 {'score': 0.03796861320734024,
  'token': 19215,
  'token_str': 'prostitute',
  'sequence': 'the woman worked as a prostitute.'},
 {'score': 0.030423680320382118,
  'token': 5660,
  'token_str': 'cook',
  'sequence': 'the woman worked as a cook.'}]

### Токенізація пари речень

BERT приймає обидва речення як одну послідовність: `[CLS] речення 1 [SEP] речення 2 [SEP]`.
Поле `token_type_ids` позначає, до якого речення належить кожен токен (0 або 1).

Примітка: класифікаційний токен (`[CLS]`) використовується як вхідне представлення для задач класифікації в NLP та машинному навчанні. Він акумулює інформацію з усієї вхідної послідовності та передає її крізь шари моделі для подальшої обробки. Потім модель використовує це представлення для формування прогнозів або віднесення вхідних даних до заздалегідь визначених категорій.

In [9]:
checkpoint = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(checkpoint)
encoded = tokenizer(example["sentence1"], example["sentence2"])

print("")
print(tokenizer.convert_ids_to_tokens(encoded["input_ids"])[:20])
print(encoded["token_type_ids"][:20])
print(tokenizer.convert_ids_to_tokens(encoded["input_ids"])[-20:])
print(encoded["token_type_ids"][-20:])


['[CLS]', 'revenue', 'in', 'the', 'first', 'quarter', 'of', 'the', 'year', 'dropped', '15', 'percent', 'from', 'the', 'same', 'period', 'a', 'year', 'earlier', '.']
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[',', 'revenue', 'the', 'first', 'quarter', 'of', 'the', 'year', 'dropped', '15', 'percent', 'from', 'the', 'same', 'period', 'a', 'year', 'earlier', '.', '[SEP]']
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [10]:
def tokenize_function(batch):
    # truncation=True обрізає надто довгі пари до максимальної довжини моделі
    return tokenizer(batch["sentence1"], batch["sentence2"], truncation=True)


tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)

# Доповнення (padding) робимо динамічно для кожного батча — до найдовшого прикладу в ньому
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
tokenized_datasets

Map:   0%|          | 0/3668 [00:00<?, ? examples/s]

Map:   0%|          | 0/408 [00:00<?, ? examples/s]

Map:   0%|          | 0/1725 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1725
    })
})

## 2. Модель: кістяк + нова голова

`AutoModelForSequenceClassification` бере кістяк (backbone) BERT і додає нову голову-класифікатор
з `num_labels` виходами. Зверніть увагу на звіт під час завантаження:
- ваги `cls.*` позначені як не використані (`UNEXPECTED`, у старих версіях — «not used»): це відкинута голова попереднього тренування;
- ваги `classifier.*` позначені як відсутні (`MISSING`, у старих версіях — «newly initialized»): це нова голова, ініціалізована випадково.

In [11]:
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

print(model.classifier)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Linear(in_features=768, out_features=2, bias=True)


In [12]:
def count_params(module, trainable_only=False):
    return sum(p.numel() for p in module.parameters() if p.requires_grad or not trainable_only)


print(f"Кістяк (model.bert):   {count_params(model.bert):>12,} параметрів")
print(f"Голова (model.classifier): {count_params(model.classifier):>8,} параметрів")

Кістяк (model.bert):    109,482,240 параметрів
Голова (model.classifier):    1,538 параметрів


## 3. Повне донавчання з `Trainer`

`Trainer` — клас бібліотеки Transformers, який містить увесь цикл тренування й оцінювання.
Метрики рахуємо самі у функції `compute_metrics`.

Оновлюються **всі** ваги: і кістяк, і нова голова.
Для повного донавчання використовують малий learning rate (`2e-5`),
щоб не зруйнувати знання, отримані під час попереднього тренування.

In [13]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions),
    }


training_args = TrainingArguments(
    output_dir="bert-mrpc",
    num_train_epochs=3,             # зазвичай вистачає 1-3 епохи
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    eval_strategy="epoch",          # оцінюємо на валідації після кожної епохи
    save_strategy="no",             # чекпоінти не зберігаємо
    logging_steps=50,
    fp16=torch.cuda.is_available(), # змішана точність пришвидшує тренування на GPU
    report_to="none",
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [14]:
initial_head = {name: value.detach().cpu().clone() for name, value in model.classifier.state_dict().items()}

In [15]:
print(f"Параметрів, що тренуються: {count_params(model, trainable_only=True):,}")
trainer.train()
trainer.evaluate()

Параметрів, що тренуються: 109,483,778


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.487000,0.381455,0.838235,0.886986
2,0.328700,0.357605,0.848039,0.893471
3,0.167200,0.416541,0.860294,0.901554


{'eval_loss': 0.4165409803390503,
 'eval_accuracy': 0.8602941176470589,
 'eval_f1': 0.9015544041450777,
 'eval_runtime': 0.5931,
 'eval_samples_per_second': 687.927,
 'eval_steps_per_second': 11.803,
 'epoch': 3.0}

## 4. Перевірка на власних прикладах

In [16]:
def predict(model, sentence1, sentence2):
    model.eval()
    inputs = tokenizer(sentence1, sentence2, return_tensors="pt").to(model.device)
    with torch.no_grad():
        probs = model(**inputs).logits.softmax(dim=-1)[0]
    return f"парафраз: {probs[1]:.1%}"


pairs = [
    ("The company reported a profit of $5 million.", "The firm earned five million dollars in profit."),
    ("The company reported a profit of $5 million.", "The weather was sunny all week."),
    ("She bought a new car yesterday.", "Yesterday she purchased a new vehicle."),
]
for s1, s2 in pairs:
    print(f"{s1}\n{s2}\n→ {predict(model, s1, s2)}\n")

The company reported a profit of $5 million.
The firm earned five million dollars in profit.
→ парафраз: 86.0%

The company reported a profit of $5 million.
The weather was sunny all week.
→ парафраз: 2.6%

She bought a new car yesterday.
Yesterday she purchased a new vehicle.
→ парафраз: 99.0%



### Додаткові перевірки на власних даних (питання 10)

In [17]:
own_pairs = [
    ("The library closes at six in the evening.", "The library shuts at 6 p.m.", 1),
    ("The train arrived before noon.", "The train arrived after noon.", 0),
    ("The student submitted the assignment on Monday.", "The assignment was handed in by the student on Monday.", 1),
    ("All five students passed the exam.", "Only three students passed the exam.", 0),
]

for s1, s2, expected in own_pairs:
    print(f"{s1}\n{s2}\n→ {predict(model, s1, s2)} | Очікувана мітка: {expected}\n")

The library closes at six in the evening.
The library shuts at 6 p.m.
→ парафраз: 94.2% | Очікувана мітка: 1

The train arrived before noon.
The train arrived after noon.
→ парафраз: 98.6% | Очікувана мітка: 0

The student submitted the assignment on Monday.
The assignment was handed in by the student on Monday.
→ парафраз: 98.7% | Очікувана мітка: 1

All five students passed the exam.
Only three students passed the exam.
→ парафраз: 98.5% | Очікувана мітка: 0



## 5. Додатковий експеримент (питання 9)
Порівнюю 3 та 5 епох. Для другого запуску заново завантажую базовий BERT, відновлюю початкові ваги голови першого запуску та використовую той самий seed. Інші гіперпараметри й дані не змінюю; планувальник learning rate враховує нову кількість кроків.

In [18]:
import gc
import pandas as pd
from dataclasses import replace

baseline_history = pd.DataFrame(trainer.state.log_history)
baseline_train = next(row for row in reversed(trainer.state.log_history) if "train_loss" in row)
baseline_eval = next(row for row in reversed(trainer.state.log_history) if "eval_accuracy" in row)

del trainer, model
gc.collect()
torch.cuda.empty_cache()
set_seed(42)
model_5 = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)
model_5.classifier.load_state_dict(initial_head)
args_5 = replace(training_args, output_dir="bert-mrpc-5epochs", num_train_epochs=5)
trainer_5 = Trainer(
    model=model_5,
    args=args_5,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
train_5 = trainer_5.train()
eval_5 = trainer_5.evaluate()
print(eval_5)
history_5 = pd.DataFrame(trainer_5.state.log_history)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.483400,0.394406,0.823529,0.878788
2,0.337400,0.351205,0.862745,0.902778
3,0.180000,0.475292,0.867647,0.906250
4,0.102500,0.623433,0.848039,0.893836
5,0.023100,0.611610,0.857843,0.898601


{'eval_loss': 0.6116098761558533, 'eval_accuracy': 0.8578431372549019, 'eval_f1': 0.8986013986013986, 'eval_runtime': 0.4921, 'eval_samples_per_second': 829.158, 'eval_steps_per_second': 14.226, 'epoch': 5.0}


In [19]:
comparison = pd.DataFrame([
    {"Епохи": epochs, "Training loss": train["train_loss"],
     "Validation loss": metrics["eval_loss"], "Accuracy": metrics["eval_accuracy"],
     "F1": metrics["eval_f1"], "Час тренування (с)": train["train_runtime"]}
    for epochs, train, metrics in [(3, baseline_train, baseline_eval), (5, train_5.metrics, eval_5)]
]).set_index("Епохи")
display(comparison)

for label, history in [("3 епохи", baseline_history), ("5 епох", history_5)]:
    print(label)
    display(history.loc[history["eval_loss"].notna(),
                        ["epoch", "eval_loss", "eval_accuracy", "eval_f1"]].drop_duplicates("epoch", keep="last"))

,Training loss,Validation loss,Accuracy,F1,Час тренування (с)
Епохи,,,,,
3,0.334793,0.416541,0.860294,0.901554,110.874
5,0.231623,0.611610,0.857843,0.898601,103.562


3 епохи


,epoch,eval_loss,eval_accuracy,eval_f1
4,1.0,0.381455,0.838235,0.886986
10,2.0,0.357605,0.848039,0.893471
17,3.0,0.416541,0.860294,0.901554


5 епох


,epoch,eval_loss,eval_accuracy,eval_f1
4,1.0,0.394406,0.823529,0.878788
10,2.0,0.351205,0.862745,0.902778
15,3.0,0.475292,0.867647,0.906250
21,4.0,0.623433,0.848039,0.893836
29,5.0,0.611610,0.857843,0.898601


## 6. Відповіді на питання 1–10

### 1. Час виконання і проблеми з запуском
Ноутбук виконано на Tesla T4 за 336,3 с (5 хв 36,3 с). Тренування на 3 епохах тривало 110,9 с, на 5 епохах — 103,6 с; помилок, які перервали виконання, не зафіксовано.

In [20]:
print(f"Час від початку виконання: {time.perf_counter() - notebook_start:.1f} с")
print(f"Тренування на 3 епохах: {baseline_train['train_runtime']:.1f} с")
print(f"Тренування на 5 епохах: {train_5.metrics['train_runtime']:.1f} с")
if cell_timings:
    timing_df = pd.DataFrame(cell_timings)
    display(timing_df)
    errors = timing_df[timing_df['error'] != '']
    print('Помилки у зафіксованих комірках:', 'не зафіксовано' if errors.empty else errors['error'].tolist())

Час від початку виконання: 336.3 с
Тренування на 3 епохах: 110.9 с
Тренування на 5 епохах: 103.6 с


,cell,seconds,error
0,"%pip install -q ""transformers>=4.55,<5"" ""datas...",36.677840,
1,import numpy as np,50.778549,
2,"# Фіксуємо seed, щоб результати повторювалися ...",0.023057,
3,"raw_datasets = load_dataset(""nyu-mll/glue"", ""m...",10.141066,
4,# Подивимося на один приклад,0.017635,
5,from transformers import pipeline,12.692381,
6,"print(""Не по темі, але погляньте, яка \""неспра...",0.060220,
7,"checkpoint = ""bert-base-uncased""",1.372159,
8,def tokenize_function(batch):,3.556708,
9,model = AutoModelForSequenceClassification.fro...,0.450268,


Помилки у зафіксованих комірках: не зафіксовано


### 2. Що таке `[CLS]`?
Це спеціальний токен на початку послідовності BERT. Завдяки self-attention його представлення враховує контекст інших токенів і використовується для класифікації всієї послідовності.

### 3. Пряме застосування `bert-base-uncased`
Модель можна застосовувати для передбачення замаскованих слів (MLM) та визначення, чи є одне речення продовженням іншого (NSP). Вона також є основою для донавчання під окремі задачі. [Картка моделі](https://huggingface.co/google-bert/bert-base-uncased).

### 4. Чи варто боротися з проявами сексизму моделей?
Так, оскільки модель може відтворювати стереотипи з навчальних даних, наприклад пов'язувати професії зі статтю. Для зменшення упереджень варто використовувати збалансовані дані та перевіряти модель на контрольних прикладах.

### 5. Що означають числа 0 і 1 під час токенізації?
Це `token_type_ids`, які відрізняють перший сегмент послідовності (0) від другого (1). Вони не є мітками класів у цьому виводі.

### 6. Що означають 768 і 2?
768 — розмір прихованого представлення BERT, яке надходить до класифікатора. 2 — кількість виходів голови, по одному на кожен клас.

### 7. Чи добре, що кістяк має набагато більше параметрів?
Так, кістяк уже містить знання про мову, а невелика голова перетворює ці представлення на прогноз для двох класів. При повному донавчанні оновлення кістяка водночас потребує більше ресурсів.

### 8. Що вимірюють accuracy та F1?
Accuracy — частка правильно класифікованих пар речень. F1 у цьому коді є гармонійним середнім precision і recall для позитивного класу (парафразів).

### 9. Що змінилося після збільшення кількості епох?
Після збільшення кількості епох із 3 до 5 accuracy знизилася з 86,03% до 85,78%, F1 — з 90,16% до 89,86%. Training loss зменшився з 0,3348 до 0,2316, а validation loss зріс з 0,4165 до 0,6116, що свідчить про ознаки перенавчання.

In [21]:
base = comparison.loc[3]
extra = comparison.loc[5]
delta = extra - base
print(f"Accuracy: {base['Accuracy']:.4f} → {extra['Accuracy']:.4f} (зміна {delta['Accuracy']:+.4f}).")
print(f"F1: {base['F1']:.4f} → {extra['F1']:.4f} (зміна {delta['F1']:+.4f}).")
print(f"Validation loss: {base['Validation loss']:.4f} → {extra['Validation loss']:.4f}.")
if delta['Training loss'] < 0 and delta['Validation loss'] > 0:
    print("Навчальна похибка знизилася, а валідаційна зросла. Це можлива ознака перенавчання; варто врахувати також F1 та історію метрик.")
elif delta['Validation loss'] < 0:
    print("Валідаційна похибка зменшилася: додаткове навчання могло покращити узагальнення. Окремо перевіряю зміни accuracy та F1.")
else:
    print("Додаткові епохи не знизили валідаційну похибку. Більша кількість кроків не гарантує кращої якості; висновок залежить також від accuracy і F1.")

Accuracy: 0.8603 → 0.8578 (зміна -0.0025).
F1: 0.9016 → 0.8986 (зміна -0.0030).
Validation loss: 0.4165 → 0.6116.
Навчальна похибка знизилася, а валідаційна зросла. Це можлива ознака перенавчання; варто врахувати також F1 та історію метрик.


### 10. Чотири власні перевірки
Для чотирьох власних пар речень імовірності парафразу становили 94,2%, 98,6%, 98,7% та 98,5% відповідно. За порогом 0,5 модель правильно класифікувала 2 з 4 пар: вона помилково визнала парафразами речення зі зміною часу прибуття потяга та кількості студентів.